# Lab 3 · Classifying forest types

**Day 2 · about 45 minutes · Student notebook**

> **Goal.** Train a Random Forest to separate evergreen from deciduous forest, using the fact that deciduous trees drop their leaves in the dry season — then measure how well it worked.

---

### How to work in this notebook

Each step gives you a **prompt card**. Copy it into Claude (or Gemini in Colab),
paste the code you get back into the empty cell below it, and run it.
Then read the **check** underneath and make sure your numbers are believable.

If the AI's code throws an error: copy the **whole** error message, paste it back to the
assistant with the sentence *"this is the error I got, please fix the code"*, and try again.
Do not retype the code by hand.


## The idea, before any code

You cannot tell an evergreen forest from a deciduous forest in a single satellite image.
They both look green in the wet season.

But watch them **across a year** and they separate immediately: deciduous forest drops its
leaves in the dry season and its NDVI collapses. Evergreen forest barely changes.

So our plan is:

1. Build a **wet season** composite and a **dry season** composite.
2. Compute `NDVI_wet − NDVI_dry`. Call it **NDVI amplitude**.
3. Low amplitude → evergreen. High amplitude → deciduous.
4. Use that rule to auto-label training examples, train a Random Forest on the full spectral
   data, and check the accuracy.

Step 3 is a *rule*. Step 4 turns it into a *model* that uses all the bands, not just NDVI.

### Setup

In [ ]:
# Run this first, every session. Colab forgets everything when it recycles.
!pip install -q geemap

import ee, geemap

ee.Authenticate()                      # opens a link - sign in, paste the code back
ee.Initialize(project='YOUR-PROJECT-ID')   # <-- put YOUR project ID here

print('Earth Engine is ready.')

In [ ]:
# Study area: one Thai province. No shapefile upload needed.
PROVINCE = 'Nan'        # <-- change to your own province

aoi_fc = (ee.FeatureCollection('FAO/GAUL/2015/level1')
          .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))
          .filter(ee.Filter.eq('ADM1_NAME', PROVINCE)))
aoi = aoi_fc.geometry()

area_ha = aoi.area(maxError=100).divide(1e4).getInfo()
print(f'{PROVINCE}: {area_ha:,.0f} hectares')

In [ ]:
BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']

def mask_s2(img):
    """Drop cloud, shadow and cirrus pixels using the SCL band, then scale to reflectance."""
    scl = img.select('SCL')
    good = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return img.updateMask(good).divide(10000)

def s2_composite(start, end, max_cloud=30):
    """Median composite over a date range - the standard way to get a cloud-free picture."""
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(aoi)
            .filterDate(start, end)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_cloud))
            .map(mask_s2)
            .median()
            .select(BANDS))

### Step 1 — two seasons

#### 🤖 Prompt card — Two seasonal composites and their NDVI difference

Copy everything in the box into your AI assistant, then paste its answer into the empty cell below.

```text
PURPOSE   Build a dry-season and a wet-season image of my area, and map the difference in
          greenness between them, so I can see which forest drops its leaves.
RESOURCE  Use COPERNICUS/S2_SR_HARMONIZED. A helper `s2_composite(start, end, max_cloud)`
          already exists and returns a cloud-masked median composite.
OUTLINE   Dry season 2025-02-01 to 2025-04-15. Wet season 2024-08-01 to 2024-10-15.
          Area is `aoi`.
MUST      Compute NDVI for each season from B8 and B4, then amplitude = NDVI_wet - NDVI_dry.
          Show all three on a geemap Map with a sensible palette.
PLATFORM  Google Colab, `ee` and `geemap` imported and authenticated.
TEST      Display the map so I can toggle between the two seasons.
```

In [ ]:
# Paste the code your AI assistant gave you here, then run it.


Look at the **NDVI amplitude** layer. Green areas barely changed between seasons — evergreen.
Orange areas changed a lot — deciduous. You can see the forest types with your own eyes now.

### Step 2 — is the signal actually there?

Before trusting the idea, measure it. If evergreen and deciduous really differ, the amplitude values over forest should be *spread out*, not clustered at one value.

In [ ]:
tree_mask = ee.Image('ESA/WorldCover/v200/2021').select('Map').eq(10)

pct = amplitude.updateMask(tree_mask).reduceRegion(
    ee.Reducer.percentile([5, 25, 50, 75, 95]),
    geometry=aoi, scale=200, maxPixels=1e9, bestEffort=True).getInfo()

for k in sorted(pct, key=lambda x: int(x.split('p')[-1])):
    print(f'{k:>16} : {pct[k]:.3f}')

> ### ✓ Check your answer
>
> For Nan: p5 = −0.019, p25 = 0.074, **p50 = 0.160**, p75 = 0.262, p95 = 0.394.
> 
> The spread from 0.07 to 0.26 across the middle half of the forest is exactly what we hoped for —
> there really are two different kinds of forest here. If every percentile had been within 0.02 of
> the median, this whole approach would not work and we should stop.

### Step 3 — training data, made automatically

A Random Forest needs labelled examples. We generate them with the amplitude rule plus
WorldCover, then sample them evenly across classes.

> **Why not draw points by hand?** You can, and in real work you should — draw them in `geemap`
> with the marker tool. We are auto-labelling here so that all 30 of us get the same answer in
> the time we have. The trade-off is real: our "truth" is only as good as the rule that made it.

In [ ]:
wc = ee.Image('ESA/WorldCover/v200/2021').select('Map')

seed = (ee.Image(0)
        .where(wc.eq(10).And(amplitude.lt(0.08)), 1)   # 1 evergreen  - little seasonal change
        .where(wc.eq(10).And(amplitude.gt(0.20)), 2)   # 2 deciduous  - big leaf-off drop
        .where(wc.eq(40), 3)                            # 3 cropland
        .where(wc.eq(50).Or(wc.eq(80)), 4)              # 4 built or water
        .selfMask().rename('class'))

# Everything the model gets to look at: both seasons, all bands, plus the indices
stack = (dry
         .addBands(wet.rename([b + '_wet' for b in BANDS]))
         .addBands([ndvi_dry, ndvi_wet, amplitude]))
FEATURES = stack.bandNames()

samples = stack.addBands(seed).stratifiedSample(
    numPoints=300, classBand='class', region=aoi,
    scale=100, seed=42, geometries=True, tileScale=2)

print('training points:', samples.size().getInfo())

> ### ✓ Check your answer
>
> You asked for 300 points in each of 4 classes, so expect **1200**. Fewer means one class barely exists in your province — check which one before continuing.

### Step 4 — train, and hold some data back

**The single most important rule in classification:** never measure your accuracy on the same
points you trained with. The model has already seen them; it will look far better than it is.

Split 70 / 30. Train on 70. Judge on the 30 it has never seen.

#### 🤖 Prompt card — Random Forest with an honest accuracy check

Copy everything in the box into your AI assistant, then paste its answer into the empty cell below.

```text
PURPOSE   Train a Random Forest to classify forest types and tell me how accurate it really is.
RESOURCE  Training points are in `samples` (property "class"), predictors in `FEATURES`.
OUTLINE   Split 70% train / 30% test using randomColumn with seed 42.
MUST      Use ee.Classifier.smileRandomForest with 50 trees.
          Compute the confusion matrix on the TEST set only, not the training set.
          Print overall accuracy and kappa.
PLATFORM  Google Colab, Earth Engine Python API.
TEST      Print the confusion matrix as an array so I can see which classes get confused.
```

In [ ]:
# Paste the code your AI assistant gave you here, then run it.


> ### ✓ Check your answer
>
> Expect **overall accuracy around 0.86 and kappa around 0.82** for Nan.
> 
> **Read the confusion matrix, do not just read the accuracy.** In our run, cropland was the
> weakest class — it gets mixed up with deciduous forest, because a harvested field and a
> leafless forest look similar from space in April. That is a real limitation of this method,
> and you should say so if you present this map.
> 
> If your accuracy comes out **above 0.98**, be suspicious rather than pleased. It almost always
> means the test points leaked into training.

### Step 5 — the map, and the areas

In [ ]:
classified = stack.classify(rf).clip(aoi)

PALETTE = ['#1F3D1C', '#8FBF6A', '#E8C468', '#6E9EC4']   # evergreen, deciduous, crop, water/built
Map.addLayer(classified, {'min': 1, 'max': 4, 'palette': PALETTE}, 'Forest type')
Map.add_legend(title='Forest type',
               legend_dict={NAMES[i]: PALETTE[i-1] for i in sorted(NAMES)})

areas = (ee.Image.pixelArea().divide(1e4).addBands(classified)
         .reduceRegion(ee.Reducer.sum().group(groupField=1, groupName='class'),
                       geometry=aoi, scale=200, maxPixels=1e9, bestEffort=True).getInfo())

print(f"{'Class':<16}{'Hectares':>14}")
print('-' * 30)
for g in sorted(areas['groups'], key=lambda x: -x['sum']):
    print(f"{NAMES.get(g['class'], '?'):<16}{g['sum']:>14,.0f}")
Map

> ### ✓ Check your answer
>
> For Nan: Deciduous **352,246 ha**, Evergreen **332,599 ha**, Cropland **152,866 ha**,
> Water/Built **29,092 ha**.
> 
> Northern Thailand really is a mixed deciduous landscape, so a near 50/50 split between the two
> forest types is the right kind of answer. If you got 95% evergreen, your amplitude thresholds
> (0.08 and 0.20) probably need adjusting for your province.
> 
> **Extension if you have time:** change `numPoints` from 300 to 50 and re-run. Watch the accuracy
> fall. That is why sample size matters.